In [1]:
# 提取案例库数据的标签
import pandas as pd
import json
import random
import re
import os

def read_json_files(folder_path):
    articles,charges,keys,facts,paths = [],[],[],[],[]
    for root, dirs, files in os.walk(folder_path):
        for file in files:
            if file.endswith(".json"):
                file_path = os.path.join(root, file)
                with open(file_path, 'r', encoding='utf-8') as f:
                    lines = f.readlines()
                    for line in lines:
                        line = json.loads(line)
                        pattern = r"根据法条第(.+?)条,被告人犯了(.+?)罪"
                        matches = re.search(pattern, line["value"])
                        if matches:
                            article = matches.group(1)
                            charge = matches.group(2)
                            articles.append(article)
                            charges.append(charge)
                            keys.append(line["key2"])
                            facts.append(line["key"])
                            paths.append(line["path_list"])
    return facts,keys,charges,articles,paths
facts,keys,charges,articles,paths = read_json_files("/root/data1/liang/self-correct-retriever/data/hera_knowbase/类案")
know_df = pd.DataFrame()
know_df["fact"],know_df["key"],know_df["charge"],know_df["article"],know_df["path"]=facts,keys,charges,articles,paths
print(len(know_df))
know_df.head(2)


1000


,fact,key,charge,article,path
0,经审理查明，原审判决书确认的证据，均经一审庭审举证、质证，证据来源合法，内容真实客观，证据之...,"[经审理查明, 原审判决书确认的证据, 一审庭审举证、质证, 证据来源合法, 内容真实客观,...",伪造公司、企业、事业单位、人民团体印章,二百八十零,"[类案, 刑法, 妨害社会管理秩序罪]"
1,克东县人民检察院指控，2016年1月1日20许，在宏博花园西侧被告人王2某经营的恒信中介公司...,"[聚众, 抽头渔利, 中华人民共和国刑法, 罪, 判处至一年六个月并处罚金]",赌博,三百零三,"[类案, 刑法, 妨害社会管理秩序罪]"


In [2]:
unique_article_classes = know_df['article'].unique()
article_id_dict = {charge_class: idx for idx, charge_class in enumerate(unique_article_classes)}
know_df["path"] = ["".join(path) for path in paths]
unique_path_classes = know_df['path'].unique()
path_id_dict = {path_class: idx for idx, path_class in enumerate(unique_path_classes)}
print(article_id_dict)
print(path_id_dict)

{'二百八十零': 0, '三百零三': 1, '三百四十一': 2, '二百九十二': 3, '三百六十四': 4, '三百五十一': 5, '三百一十零': 6, '三百一十三': 7, '三百三十八': 8, '二百九十三': 9, '三百四十五': 10, '三百一十二': 11, '三百四十二': 12, '二百七十七': 13, '三百三十六': 14, '三百四十七': 15, '三百五十六': 16, '三百五十四': 17, '三百零二': 18, '三百四十零': 19, '三百一十五': 20, '三百八十四': 21, '一百三十三': 22, '一百二十八': 23, '一百一十五': 24, '一百二十四': 25, '一百一十四': 26, '一百二十五': 27, '一百三十四': 28, '一百三十二': 29, '一百七十六': 30, '二百零九': 31, '一百四十四': 32, '一百七十二': 33, '一百七十五': 34, '一百九十六': 35, '一百六十三': 36, '一百四十一': 37, '一百五十零': 38, '二百二十四': 39, '二百一十三': 40, '二百一十四': 41, '一百四十三': 42, '一百四十零': 43, '一百五十二': 44, '一百九十三': 45, '一百六十四': 46, '一百七十一': 47, '一百七十七': 48, '二百六十七': 49, '二百七十二': 50, '二百六十六': 51, '二百七十四': 52, '二百七十六': 53, '二百六十四': 54, '二百七十一': 55, '二百七十五': 56, '二百六十三': 57, '二百六十九': 58, '二百三十六': 59, '二百三十二': 60, '二百四十五': 61, '二百三十四': 62, '二百三十三': 63, '二百三十七': 64, '二百三十八': 65, '二百六十零': 66, '二百四十四': 67, '二百四十六': 68, '二百三十五': 69, '三百九十七': 70}
{'类案刑法妨害社会管理秩序罪': 0, '类案刑法贪污贿赂罪': 1, '类案刑法危害公共安全罪': 2, '类案刑法破坏社会主义市场经济秩序罪': 3, '类案刑法侵犯财产罪

In [1]:
import torch

# 假设你有这三个张量
tensor1 = torch.tensor([[1, 2, 3],
                        [4, 5, 6]])
tensor2 = torch.tensor([[7, 8, 9],
                        [10, 11, 12]])
tensor3 = torch.tensor([[13, 14, 15],
                        [16, 17, 18]])

# 使用 torch.cat 按列拼接这三个张量
combined_tensor = torch.cat((tensor1, tensor2, tensor3), dim=1)

print(combined_tensor)


tensor([[ 1,  2,  3,  7,  8,  9, 13, 14, 15],
        [ 4,  5,  6, 10, 11, 12, 16, 17, 18]])


In [6]:
import os
import json
import pandas as pd
import dgl
from tqdm import tqdm
import torch
def read_json_files(folder_path):
    articles,charges,keys,facts,paths = [],[],[],[],[]
    for root, dirs, files in os.walk(folder_path):
        for file in files:
            if file.endswith(".json"):
                file_path = os.path.join(root, file)
                with open(file_path, 'r', encoding='utf-8') as f:
                    lines = f.readlines()
                    for line in lines:
                        line = json.loads(line)
                        # pattern = r"根据法条第(.+?)条,被告人犯了(.+?)罪"
                        # matches = re.search(pattern, line["value"])
                        # if matches:
                            # article = matches.group(1)
                            # charge = matches.group(2)
                        article = line["art"]
                        charge = line["char"]
                        articles.append(article)
                        charges.append(charge)
                        keys.append(line["key2"])
                        facts.append(line["key"])
                        paths.append(line["path_list"])
    return facts,keys,charges,articles,paths
facts,keys,charges,articles,paths = read_json_files("/root/data1/liang/self-correct-retriever/data/hera_knowbase3/法院观点")
know_df = pd.DataFrame()
know_df["fact"],know_df["key"],know_df["charge"],know_df["article"],know_df["path"]=facts,keys,charges,articles,paths
know_df["path"] = ["".join(path) for path in paths]
unique_path_classes = know_df['path'].unique()
path_id_dict = {path_class: idx for idx, path_class in enumerate(unique_path_classes)}

graph = dgl.DGLGraph()  # 创建一个空图
nodes_type_A = [i for i in range(len(know_df))]
nodes_type_B = [i for i in range(len(know_df))]
# nodes_type_C = [i for i in range(len(path_id_dict))]
nodes_type_C = [i for i in range(len(know_df))]

node_types = [0] * len(nodes_type_A) + [1] * len(nodes_type_B) + [2]*len(nodes_type_C) # 节点类型标签

# 创建图，并添加节点以及节点类型信息
graph.add_nodes(len(nodes_type_A) + len(nodes_type_B)+len(nodes_type_C))
graph.ndata['node_type'] = torch.tensor(node_types)

# 假设有不同类型节点之间的连接关系，这里是随机连接的示例
edges = []
edges_set = set()
for i in tqdm(range(len(know_df))):
    article_value = know_df['article'][i]
    path_value = know_df['path'][i]
    node_C_id = path_id_dict[path_value]
    edges.append((i,i+len(know_df)))
    edges.append((i+len(know_df),i))
    # edges.append((i,len(know_df)*2+node_C_id))
    # edges.append((len(know_df)*2+node_C_id,i))
    edges.append((i,i+len(know_df)*2))
    edges.append((i+len(know_df)*2,i))
    indices = know_df[know_df['article'] == article_value].index.tolist()
    for j in indices:
        if j!=i:
            edges_set.add((i, j))
            edges.append((i,j+len(know_df)))
            edges.append((j+len(know_df),i))
            edges.append((i,j+len(know_df)*2))
            edges.append((j+len(know_df)*2,i))
edges.extend(list(edges_set))
src, dst = zip(*edges)
graph.add_edges(src, dst)

/home/ubuntu/anaconda3/envs/baichuan/lib/python3.8/site-packages/dgl/heterograph.py:92: DGLWarning: Recommend creating graphs by `dgl.graph(data)` instead of `dgl.DGLGraph(data)`.
  dgl_warning(
100%|██████████| 6124/6124 [00:05<00:00, 1106.23it/s]
